# 📒 Notebook 4 — KNN (User-Based Collaborative Filtering) Model

**Algorithm:** KNNWithMeans (K-Nearest Neighbors) via the `Surprise` library

**How KNN works (simple explanation):**
- For a given user, find the K most similar users (based on rating history)
- Look at what those similar users rated highly
- Recommend those restaurants to our user
- 'WithMeans' = adjusts for the fact that some users always rate high or always rate low

**KNN vs SVD:**
- KNN is more interpretable ('users like you loved this place')
- SVD is generally more accurate but is a black box
- We combine both in Notebook 5

**Requires:** Run Notebook 2 first to generate `data/` folder

## 1. Install Dependencies

In [2]:
#!pip install scikit-surprise pandas tqdm -q

## 2. Load Data

In [4]:
import pandas as pd
import pickle

train_df      = pd.read_csv("data/ratings_train.csv")
test_df       = pd.read_csv("data/ratings_test.csv")
business_meta = pd.read_csv("data/business_meta.csv")

with open("data/user_encoder.pkl", "rb") as f:
    user_encoder = pickle.load(f)
with open("data/business_encoder.pkl", "rb") as f:
    biz_encoder = pickle.load(f)

print(f"✅ Train: {len(train_df):,}  |  Test: {len(test_df):,}")
print(f"   Unique users     : {train_df['user_id'].nunique():,}")
print(f"   Unique businesses: {train_df['business_id'].nunique():,}")

✅ Train: 1,579,941  |  Test: 394,986
   Unique users     : 77,443
   Unique businesses: 33,016


## 3. Build Surprise Dataset

In [5]:
from surprise import Dataset, Reader, KNNWithMeans

# 🔹 Step 1: Filter active users
min_user_interactions = 20
user_counts = train_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= min_user_interactions].index

df_filtered = train_df[train_df['user_id'].isin(valid_users)]

# 🔹 Step 2: Filter active items
min_item_interactions = 20
item_counts = df_filtered['business_id'].value_counts()
valid_items = item_counts[item_counts >= min_item_interactions].index

df_filtered = df_filtered[df_filtered['business_id'].isin(valid_items)]

# 🔹 Step 3: Sample
train_df_sample = df_filtered.sample(frac=0.3, random_state=42)

print(f"Filtered + Sampled rows: {len(train_df_sample):,}")

# 🔹 Step 4: Build Surprise dataset
reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_df_sample[["user_id", "business_id", "target_rating"]],
    reader
)

trainset = train_data.build_full_trainset()

# 🔹 Step 5: Info
print(f"✅ Surprise trainset built (FILTERED SAMPLE)")
print(f"   Users  : {trainset.n_users:,}")
print(f"   Items  : {trainset.n_items:,}")
print(f"   Ratings: {trainset.n_ratings:,}")

Filtered + Sampled rows: 229,951
✅ Surprise trainset built (FILTERED SAMPLE)
   Users  : 20,975
   Items  : 13,598
   Ratings: 229,951


## 4. Train KNNWithMeans Model

In [6]:
import time

# KNN hyperparameters
# k         : number of neighbors to consider
# sim_options:
#   name    : similarity metric — 'pearson_baseline' is best for ratings
#   user_based: True = compare users, False = compare items (item-based CF)
#               User-based: 'people like you liked this'
#               Item-based:  'people who liked X also liked Y'
#   We use user-based here for interpretability

knn_model = KNNWithMeans(
    k=40,
    min_k=3,
    sim_options={
        "name":       "pearson_baseline",
        "user_based": True
    },
    verbose=True
)

print("🏋️ Training KNN model (computing user similarities — may take a few minutes)...")
start = time.time()
knn_model.fit(trainset)
elapsed = time.time() - start
print(f"✅ Training complete in {elapsed:.1f}s")

🏋️ Training KNN model (computing user similarities — may take a few minutes)...
Estimating biases using als...
Computing the pearson_baseline similarity matrix...
Done computing similarity matrix.
✅ Training complete in 37.1s


## 5. Evaluate on Test Set

In [8]:
from surprise import accuracy

testset = list(zip(
    test_df["user_id"].tolist(),
    test_df["business_id"].tolist(),
    test_df["target_rating"].tolist()
))

predictions = knn_model.test(testset)

rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)

print("=" * 40)
print("KNN Model Evaluation")
print("=" * 40)
print(f"RMSE : {rmse:.4f}  (lower is better, target < 1.0)")
print(f"MAE  : {mae:.4f}   (mean absolute error in stars)")

# Sample predictions
print("\nSample predictions (actual vs predicted):")
for pred in predictions[:5]:
    print(f"  User: {pred.uid[:8]}... | Biz: {pred.iid[:8]}... | "
          f"Actual: {pred.r_ui} | Predicted: {pred.est:.2f}")

KNN Model Evaluation
RMSE : 1.1961  (lower is better, target < 1.0)
MAE  : 0.9290   (mean absolute error in stars)

Sample predictions (actual vs predicted):
  User: qUF5K_LH... | Biz: aHHK_n_w... | Actual: 5 | Predicted: 3.90
  User: d__nxnRq... | Biz: TOgkopnD... | Actual: 1 | Predicted: 4.56
  User: cF159aEF... | Biz: zewTzLyi... | Actual: 4 | Predicted: 4.00
  User: BKp3iEfF... | Biz: -dzKS5d4... | Actual: 4 | Predicted: 3.90
  User: ixtKgePF... | Biz: w7kKNELh... | Actual: 4 | Predicted: 3.80


## 6. Compare SVD vs KNN Metrics

In [9]:
import os

# Load SVD metrics if available
print("=" * 50)
print("Model Comparison")
print("=" * 50)
print(f"{'Model':<10} {'RMSE':>8} {'MAE':>8}")
print("-" * 30)
print(f"{'KNN':<10} {rmse:>8.4f} {mae:>8.4f}")
print("\n(Compare with SVD metrics from Notebook 3)")
print("Lower RMSE/MAE = better accuracy")
print("Both models are used together in Notebook 5 (Hybrid)")

Model Comparison
Model          RMSE      MAE
------------------------------
KNN          1.1961   0.9290

(Compare with SVD metrics from Notebook 3)
Lower RMSE/MAE = better accuracy
Both models are used together in Notebook 5 (Hybrid)


## 7. Save KNN Model

In [10]:
import pickle, os
os.makedirs("models", exist_ok=True)

with open("models/knn_model.pkl", "wb") as f:
    pickle.dump(knn_model, f)

print("✅ KNN model saved to models/knn_model.pkl")

✅ KNN model saved to models/knn_model.pkl


## 8. Test: Get Top-3 Recommendations for an Existing User

In [11]:
def knn_recommend(user_id, city=None, state=None, top_k=3):
    """
    Get top-K restaurant recommendations for an existing user using KNN.
    
    Args:
        user_id : The user's string ID from the database
        city    : Optional city filter (e.g. 'Philadelphia')
        state   : Optional state filter (e.g. 'PA')
        top_k   : Number of recommendations to return
    """
    if user_id not in user_encoder["str2idx"]:
        print(f"⚠️  User '{user_id}' not found in training data. Use Notebook 5 for new users.")
        return []

    # Restaurants already rated by this user
    rated_biz = set(train_df[train_df["user_id"] == user_id]["business_id"].tolist())

    # Filter candidates by location
    candidates = business_meta.copy()
    if city:
        candidates = candidates[candidates["city"].str.lower() == city.lower()]
    if state:
        candidates = candidates[candidates["state"].str.upper() == state.upper()]

    candidates = candidates[~candidates["business_id"].isin(rated_biz)]

    if candidates.empty:
        print(f"⚠️  No unrated businesses found for city={city}, state={state}")
        return []

    # Predict ratings
    preds = []
    for _, biz in candidates.iterrows():
        pred = knn_model.predict(user_id, biz["business_id"])
        preds.append({
            "business_id":   biz["business_id"],
            "business_name": biz["business_name"],
            "city":          biz["city"],
            "state":         biz["state"],
            "avg_stars":     biz["business_avg_stars"],
            "categories":    biz["categories"],
            "knn_score":     pred.est
        })

    preds_df = pd.DataFrame(preds).sort_values("knn_score", ascending=False).head(top_k)

    print(f"\n🎯 KNN Top-{top_k} for user '{user_id[:12]}...' in {city or 'any city'}, {state or 'any state'}")
    print("─" * 65)
    for i, (_, row) in enumerate(preds_df.iterrows()):
        print(f"#{i+1} {row['business_name']} ({row['city']}, {row['state']})")
        print(f"   Categories : {row['categories']}")
        print(f"   Avg Stars  : ⭐{row['avg_stars']}")
        print(f"   KNN Score  : {row['knn_score']:.3f}")
    return preds_df


# Test with real user
sample_user = train_df["user_id"].iloc[0]
print(f"Testing with user: {sample_user}")
knn_recommend(sample_user, top_k=3)

Testing with user: 6jz_Yr6_AP2WWLbj9gGDpA

🎯 KNN Top-3 for user '6jz_Yr6_AP2W...' in any city, any state
─────────────────────────────────────────────────────────────────
#1 Jack Dempsey's Restaurant & Bar (New Orleans, LA)
   Categories : Seafood, Restaurants
   Avg Stars  : ⭐4.0
   KNN Score  : 5.000
#2 McClure's (New Orleans, LA)
   Categories : Tapas/Small Plates, Restaurants, Burgers, Barbeque
   Avg Stars  : ⭐4.0
   KNN Score  : 4.723
#3 Pho Tau Bay Restaurant (New Orleans, LA)
   Categories : Restaurants, Event Planning & Services, Chinese, Caterers, Vietnamese, Breakfast & Brunch
   Avg Stars  : ⭐4.0
   KNN Score  : 4.612


,business_id,business_name,city,state,avg_stars,categories,knn_score
9653,xJ3NSwE0xhdtA-tB_y_rNQ,Jack Dempsey's Restaurant & Bar,New Orleans,LA,4.0,"Seafood, Restaurants",5.000000
3104,Ye5nmTmZ7cL7BQshqZeIWQ,McClure's,New Orleans,LA,4.0,"Tapas/Small Plates, Restaurants, Burgers, Barb...",4.722981
5772,x4K6aMaOYvGhC5jhFJP2Ag,Pho Tau Bay Restaurant,New Orleans,LA,4.0,"Restaurants, Event Planning & Services, Chines...",4.612166


## 9. Test with City/State Filter

In [12]:
sample_city  = business_meta["city"].value_counts().index[0]
sample_state = business_meta[business_meta["city"] == sample_city]["state"].iloc[0]
print(f"Testing city filter: {sample_city}, {sample_state}")

knn_recommend(sample_user, city=sample_city, state=sample_state, top_k=3)

Testing city filter: Philadelphia, PA

🎯 KNN Top-3 for user '6jz_Yr6_AP2W...' in Philadelphia, PA
─────────────────────────────────────────────────────────────────
#1 Honey's Sit-N-Eat (Philadelphia, PA)
   Categories : Southern, Restaurants, American (Traditional)
   Avg Stars  : ⭐4.0
   KNN Score  : 4.488
#2 IndeBlue Modern Indian Food & Spirits (Philadelphia, PA)
   Categories : Cocktail Bars, Food Delivery Services, Nightlife, Breakfast & Brunch, Food, Bars, Event Planning & Services, Caterers, Restaurants, Indian
   Avg Stars  : ⭐4.5
   KNN Score  : 4.400
#3 Root (Philadelphia, PA)
   Categories : Wine Bars, Nightlife, Bars, American (New), Restaurants, Italian, Spanish
   Avg Stars  : ⭐4.5
   KNN Score  : 4.400


,business_id,business_name,city,state,avg_stars,categories,knn_score
573,cXSyVvOr9YRN9diDkaWs0Q,Honey's Sit-N-Eat,Philadelphia,PA,4.0,"Southern, Restaurants, American (Traditional)",4.487623
0,-0TffRSXXIlBYVbb5AwfTg,IndeBlue Modern Indian Food & Spirits,Philadelphia,PA,4.5,"Cocktail Bars, Food Delivery Services, Nightli...",4.400000
1494,JXYKzNSaK_fznAXvmRdYPA,Root,Philadelphia,PA,4.5,"Wine Bars, Nightlife, Bars, American (New), Re...",4.400000
